In [82]:
import pandas as pd
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from PIL import Image
from torch.utils.data import DataLoader
import torch.nn.functional as F

In [83]:
device=("cuda" if torch.cuda.is_available() else "cpu")

In [84]:
device

'cpu'

In [86]:
train_df=pd.DataFrame(columns=["img_name","label"])
train_df["img_name"]=os.listdir("train_new/")
for idx, i in enumerate(os.listdir("train_new/")):
    if "cat" in i:
        train_df["label"][idx]=0
    elif "dog" in i:
        train_df["label"][idx]=1

In [87]:
train_df.to_csv(r'train_csv.csv',index=False,header=True)

In [307]:
## PIL is a popular computer vision library that allows us to load images in python and convert it to RGB format.
## Objective:- Use images from train folder and the image filenames, labels from train_df to return (image,label) tuple 


class CatsAndDogsDataset(Dataset):
    def __init__(self,root_dir,annotation_file,transform=None):
        self.root_dir=root_dir
        self.annotations=pd.read_csv(annotation_file) 
        self.to_tensor=ToTensor()
        self.norm=Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        
    def __len__(self):
        return len(self.annotations)   ## return number of entries in train_df
    
    def __getitem__(self,index):
        img_id = self.annotations.iloc[index, 0]  ## name of the image file, train_df.iloc[index,"img_name"] "img_name" is 0
        ## os.path.join uses the “/” symbol to combine the root_dir(“train/”) and img_name(image file name) = train/cat.0.jpg example
        ## PIL is used to load the image and convert it to RGB format
        img = Image.open(os.path.join(self.root_dir, img_id)).convert("RGB")  
        y_label = torch.tensor(float(self.annotations.iloc[index, 1])) ## 1 is "label" y_label of an image extracted and converteed to tensor

        img_tensor = (torch.from_numpy(np.array(img)).permute(2,0,1)).float()/255.0

        img = F.interpolate(img_tensor.unsqueeze(0), size=(400,500),align_corners=None)
        return (img, y_label)

In [419]:
dataset = CatsAndDogsDataset("train_new","train_csv.csv")

train_set, test_set = torch.utils.data.random_split(dataset,[31,10])
#,pin_memory=True
train_loader = DataLoader(dataset=train_set, shuffle=True, batch_size=2,num_workers=0)

test_loader = DataLoader(dataset=test_set, shuffle=True, batch_size=2,num_workers=0)

In [420]:
# out_channels,kernel_size,out_features are chosen manually, they are chosen arbitrarily, it is the k=job of the designer to choose its values

In [421]:
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 97 * 122, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

In [422]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [423]:
for i, data in enumerate(train_loader, 0):
    # get the inputs; data is a list of [inputs, labels]
    inputs, labels = data
    print(inputs.shape)

torch.Size([2, 1, 3, 400, 500])
torch.Size([2, 1, 3, 400, 500])
torch.Size([2, 1, 3, 400, 500])
torch.Size([2, 1, 3, 400, 500])
torch.Size([2, 1, 3, 400, 500])
torch.Size([2, 1, 3, 400, 500])
torch.Size([2, 1, 3, 400, 500])
torch.Size([2, 1, 3, 400, 500])
torch.Size([2, 1, 3, 400, 500])
torch.Size([2, 1, 3, 400, 500])
torch.Size([2, 1, 3, 400, 500])
torch.Size([2, 1, 3, 400, 500])
torch.Size([2, 1, 3, 400, 500])
torch.Size([2, 1, 3, 400, 500])
torch.Size([2, 1, 3, 400, 500])
torch.Size([1, 1, 3, 400, 500])


In [424]:
for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data
        
        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs.squeeze(1))
        loss = criterion(outputs, labels.type(torch.LongTensor))
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
#         if i % 2 == 1:    
        print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2:.3f}')
        running_loss = 0.0

print('Finished Training')

[1,     1] loss: 0.311
[1,     2] loss: 0.347
[1,     3] loss: 0.348
[1,     4] loss: 0.308
[1,     5] loss: 0.349
[1,     6] loss: 0.348
[1,     7] loss: 0.304
[1,     8] loss: 0.397
[1,     9] loss: 0.301
[1,    10] loss: 0.349
[1,    11] loss: 0.298
[1,    12] loss: 0.350
[1,    13] loss: 0.349
[1,    14] loss: 0.407
[1,    15] loss: 0.350
[1,    16] loss: 0.405
[2,     1] loss: 0.349
[2,     2] loss: 0.348
[2,     3] loss: 0.348
[2,     4] loss: 0.348
[2,     5] loss: 0.347
[2,     6] loss: 0.346
[2,     7] loss: 0.345
[2,     8] loss: 0.348
[2,     9] loss: 0.314
[2,    10] loss: 0.377
[2,    11] loss: 0.315
[2,    12] loss: 0.316
[2,    13] loss: 0.346
[2,    14] loss: 0.346
[2,    15] loss: 0.311
[2,    16] loss: 0.385
Finished Training


In [425]:
PATH = './cifar_net.pth'
torch.save(net.state_dict(), PATH)

In [426]:
dataiter = iter(test_loader)
images, labels = next(dataiter)

In [427]:
net = Net()
net.load_state_dict(torch.load(PATH, weights_only=True))

<All keys matched successfully>

In [431]:
outputs = net(images.squeeze(1))

In [432]:
classes = ('cat', 'dog')

In [433]:
_, predicted = torch.max(outputs, 1)

print('Predicted: ', ' '.join(f'{classes[predicted[j]]:5s}'
                              for j in range(2)))

Predicted:  cat   cat  


In [434]:
correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in test_loader:
        images, labels = data
        # calculate outputs by running images through the network
        outputs = net(images.squeeze(1))
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10 test images: {100 * correct // total} %')

Accuracy of the network on the 10 test images: 40 %


In [439]:
total_pred[classes[1]]

0

In [444]:
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

# again no gradients needed
with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = net(images.squeeze(1))
        _, predictions = torch.max(outputs, 1)
        # collect the correct predictions for each class
        for label, prediction in zip(labels, predictions):
            label=label.to(dtype=torch.long)
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1


# print accuracy for each class
for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')

Accuracy for class: cat   is 100.0 %
Accuracy for class: dog   is 0.0 %


In [445]:
# def image_to_tensor(image):
#     image_np = np.array(image)
#     image_tensor = torch.from_numpy(image_np)
    
#     # Permute from (H, W, C) to (C, H, W) for PyTorch
#     image_tensor = image_tensor.permute(2, 0, 1)
    
#     # normalize
#     image_tensor = image_tensor.float() / 255.0
    
#     return image_tensor